# CLIP + SDM: Random và Balanced Mixed, 60 epochs

Bật Internet, GPU T4 ×2, gắn dataset RSTPReid. Repo private: thêm Kaggle Secret **GH_TOKEN** có quyền đọc `lqb464/clip-balanced-mixed` và cấp quyền truy cập secret cho notebook.

Batch sampler/loss luôn là **128**; microbatch encoder là 16. Mỗi GPU chạy một sampler. Chạy các cell theo thứ tự. Để tiếp tục session trước, gắn output đã lưu và điền `RESUME_FROM` trong cell cấu hình. Phải giữ cả `last.pt`, `best.pt` và cache. Chạy đến 60 epochs tổng cộng; chưa chắc hoàn tất trong một session 12 giờ.


In [ ]:
import base64
import os
from pathlib import Path
import subprocess
import sys

from kaggle_secrets import UserSecretsClient

REPO = Path('/kaggle/working/clip-balanced-mixed')
token = UserSecretsClient().get_secret('GH_TOKEN')
clone_env = os.environ.copy()
clone_env.update({
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + base64.b64encode(
        ('x-access-token:' + token).encode()).decode(),
})
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/lqb464/clip-balanced-mixed.git',
                    str(REPO)], env=clone_env, check=True)
else:
    print('Dùng code đang có tại', REPO)
del token, clone_env
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
import shutil

DATA_ROOT = '/kaggle/input/datasets/hoanggv/tbps-benchmark/benchmark'
WORK = Path('/kaggle/working/clip_experiment')
RESUME_FROM = ''  # Session mới: điền /kaggle/input/.../clip_experiment của output đã lưu.
EPOCHS = 60
SEED = 1
MICROBATCH = 16
MAX_TRAIN_HOURS = 9  # Chưa gồm thời gian cài đặt và tạo cache.

if RESUME_FROM:
    previous = Path(RESUME_FROM)
    if not (previous / 'runs').is_dir() or not (previous / 'cache').is_dir():
        raise ValueError('RESUME_FROM phải trỏ tới thư mục chứa cả runs/ và cache/')
    if not WORK.exists():
        shutil.copytree(previous, WORK)
    else:
        print('Giữ kết quả đang có trong session hiện tại; không chép đè:', WORK)
WORK.mkdir(parents=True, exist_ok=True)
CACHE = WORK / 'cache' / 'pretrained_neighbors.npz'
print('Kết quả:', WORK, '| batch loss: 128 | epochs:', EPOCHS)


In [ ]:
cache_env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
subprocess.run([sys.executable, '-u', 'build_cache.py', '--root', DATA_ROOT,
                '--output', str(CACHE)], env=cache_env, check=True)


In [ ]:
subprocess.run([
    sys.executable, '-u', 'launch.py', '--root', DATA_ROOT,
    '--runs', str(WORK / 'runs'), '--cache', str(CACHE),
    '--gpus', '0', '1', '--epochs', str(EPOCHS), '--seed', str(SEED),
    '--microbatch', str(MICROBATCH), '--max-hours', str(MAX_TRAIN_HOURS),
], check=True)


In [ ]:
subprocess.run([sys.executable, 'compare.py', '--runs', str(WORK / 'runs'),
                '--seed', str(SEED)], check=True)
print('Lưu output notebook bằng Save Version để giữ checkpoint trước khi session kết thúc.')
print('Nếu chưa đủ 60 epochs, session tiếp theo gắn output này và đặt RESUME_FROM.')
